# Trading App V2

This notebook is the live-trading shell for `optimal_trader` going forward.

- `quant-warehouse` owns historical refresh, point-in-time data reads, feature engineering, target engineering, and ThetaData option data.
- `quant-orchestrator` owns feature-family classifier training, option-ranker training, backtests, and strategy artifacts.
- `optimal_trader` owns live broker reconciliation, order planning, Streamlit leaderboard generation, and guarded order submission.

Default behavior builds plans and artifacts only. Orders are submitted exclusively from the generated Streamlit frontend after clicking the Submit Orders button.

In [ ]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 260)

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "app").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def prefer_local_repo(package_name: str, repo_dir: Path) -> None:
    repo_dir = repo_dir.resolve()
    if not repo_dir.exists():
        return
    repo_text = str(repo_dir)
    sys.path[:] = [
        entry
        for entry in sys.path
        if str(Path(entry or ".").expanduser().resolve()) != repo_text
    ]
    sys.path.insert(0, repo_text)
    loaded = sys.modules.get(package_name)
    loaded_file = Path(str(getattr(loaded, "__file__", ""))).resolve() if loaded is not None else None
    if loaded_file is not None and repo_dir not in loaded_file.parents:
        for module_name in [name for name in sys.modules if name == package_name or name.startswith(f"{package_name}.")]:
            del sys.modules[module_name]


WORKSPACE_ROOT = REPO_ROOT.parent
prefer_local_repo("quant_orchestrator", WORKSPACE_ROOT / "quant-orchestrator")
prefer_local_repo("quant_warehouse", WORKSPACE_ROOT / "quant-warehouse")
prefer_local_repo("fmpsdk", WORKSPACE_ROOT / "fmpsdk")

from app.quant_warehouse_storage import ensure_quant_warehouse_storage
from app.trading_app_v2_runtime import (
    DEFAULT_STRATEGY_SOURCES,
    alpaca_client_from_env,
    build_alpaca_equity_orders,
    build_alpaca_option_orders,
    build_latest_equity_leaderboard,
    build_option_ml_ranking_table,
    backfill_thetadata_eod_for_score_date,
    backfill_thetadata_for_oracle_trade_windows,
    build_score_date_option_ml_ranking_table,
    build_symbol_score_table,
    build_robinhood_option_orders,
    build_llm_review_orders,
    default_paths,
    write_streamlit_leaderboard_app,
)

paths = default_paths(REPO_ROOT)
paths.artifact_root.mkdir(parents=True, exist_ok=True)
paths.live_artifact_dir.mkdir(parents=True, exist_ok=True)
paths


## Environment

This notebook assumes `quant-warehouse`, `quant-orchestrator`, and `tradingagents` are already installed in the `optimal_trader` environment. Dependency installation belongs in environment setup, not in the live trading notebook.


In [ ]:
storage = ensure_quant_warehouse_storage()
display(storage.as_dict())


def optional_date(value):
    if value is None:
        return None
    text = str(value).strip()
    if not text or text.lower() in {"none", "null", "nat"}:
        return None
    return text


## Run Configuration

Shared knobs for universe screening, FMP refresh, feature engineering, target engineering, and downstream training. Feature-family and oracle-label plans are shown after warehouse data is refreshed.


In [ ]:
RUN_EQUITY_MOE_TRAINING = True
RUN_OPTION_RANKER_TRAINING = False
DATA_START = "1900-01-01"
DATA_END = ""  # Empty means latest available downstream data.
MIN_MARKET_CAP = 10_000_000_000
TOP_K = 20  # Use 10 or 20 for live candidate breadth.
MIN_LONG_SCORE = 0.50
REQUIRE_ALL_REQUESTED_FEATURE_FAMILIES = True
REFRESH_MISSING_FMP_DATA = True
REFRESH_MISSING_FMP_INCLUDE_MACRO = True
REFRESH_MISSING_FMP_INCLUDE_PRICES = True
REFRESH_MISSING_FMP_STALENESS_DAYS = 60
REFRESH_MISSING_FMP_SKIP_RECENT_HOURS = 24.0
REFRESH_MISSING_FMP_MAX_WORKERS = 8
REFRESH_THETADATA_SCORE_DATE_EOD = True
THETADATA_OPTION_SCORE_DATE = ""  # Empty means latest equity score_date.
THETADATA_REFRESH_MAX_WORKERS = 1
THETADATA_REFRESH_OVERWRITE = False
REFRESH_THETADATA_ORACLE_TRADE_WINDOWS = True
THETADATA_ORACLE_BACKFILL_WINDOW_DAYS = 7
THETADATA_ORACLE_FALLBACK_WINDOW_DAYS = 1
THETADATA_ORACLE_BACKFILL_REQUEST_SLEEP = 0.0
THETADATA_ORACLE_BACKFILL_OVERWRITE = False
RUN_BACKTESTS_IN_ORCHESTRATOR = False

ALPACA_EQUITY_ACCOUNT_PREFIX = "EQUITY"
ALPACA_OPTION_ACCOUNT_PREFIX = "OPTION"
ALPACA_LLM_ACCOUNT_PREFIX = "LLM"

OPTION_STRATEGY_ALLOCATION = 100_000.0
OPTION_BUCKET = "otm_option"
OPTION_TENOR_DAYS = 90

# Real Robinhood follows the option strategy with deeply discounted GTC limits.
ROBINHOOD_OPTION_GATE_DISCOUNT_PCT = 90.0

# Equity and LLM real-account gates remain blocked until paper PnL justifies changing them.
ROBINHOOD_EQUITY_GATE_DISCOUNT_PCT = 100.0
ROBINHOOD_LLM_GATE_DISCOUNT_PCT = 100.0

OPTION_PANEL_FOR_RANKER = paths.artifact_root / "option_candidate_panel.parquet"

display(
    {
        "artifact_root": str(paths.artifact_root),
        "equity_artifact_dir": str(paths.equity_artifact_dir),
        "option_artifact_dir": str(paths.option_artifact_dir),
        "live_artifact_dir": str(paths.live_artifact_dir),
        "data_start": DATA_START,
        "min_market_cap": MIN_MARKET_CAP,
        "refresh_missing_fmp_data": REFRESH_MISSING_FMP_DATA,
    }
)


## FMP Historical Refresh

Refresh raw FMP warehouse data before feature engineering and target engineering. This cell screens the universe first, displays the exact symbols that will be used downstream, then refreshes only those symbols in the shared Quant Warehouse store. The training cell below disables internal refresh so model fitting starts only after this visible refresh step completes.


In [ ]:
from quant_warehouse.migrate.backfill_missing_fmp import backfill_missing_fmp_historical
from quant_warehouse.research_tools.feature_family_eval import FamilyEvaluationConfig, screen_fmp_equity_universe
from quant_warehouse.warehouse.api import Warehouse

refresh_warehouse = Warehouse()
refresh_feature_config = FamilyEvaluationConfig(
    provider="fmp",
    market_cap_min=MIN_MARKET_CAP,
    start_date=DATA_START,
    end_date=optional_date(DATA_END),
)

screened_equity_symbols, raw_fmp_universe, universe_eligibility, universe_source = screen_fmp_equity_universe(
    refresh_feature_config,
    warehouse=refresh_warehouse,
)
screened_symbols_df = pd.DataFrame({"symbol": list(screened_equity_symbols)})
print(
    f"FMP universe source={universe_source}; "
    f"eligible_symbols={len(screened_equity_symbols):,}; "
    f"min_market_cap={MIN_MARKET_CAP:,}"
)
display(screened_symbols_df)
print("Universe eligibility sample")
display(universe_eligibility.head(100))


def notebook_refresh_log(message: str) -> None:
    print(f"[fmp-refresh] {message}", flush=True)


fmp_refresh_summary = {"status": "skipped_by_config", "equity_symbols": list(screened_equity_symbols), "etf_symbols": []}
if REFRESH_MISSING_FMP_DATA:
    fmp_refresh_summary = backfill_missing_fmp_historical(
        warehouse=refresh_warehouse,
        equity_provider="fmp",
        etf_provider="fmp",
        equity_symbols=screened_equity_symbols,
        etf_symbols=(),
        include_macro=REFRESH_MISSING_FMP_INCLUDE_MACRO,
        include_prices=REFRESH_MISSING_FMP_INCLUDE_PRICES,
        staleness_days=REFRESH_MISSING_FMP_STALENESS_DAYS,
        skip_recent_hours=REFRESH_MISSING_FMP_SKIP_RECENT_HOURS,
        max_workers=REFRESH_MISSING_FMP_MAX_WORKERS,
        progress_logger=notebook_refresh_log,
    )
else:
    print("FMP refresh skipped because REFRESH_MISSING_FMP_DATA=False")

def macro_refresh_status(macro_payload) -> str | None:
    if isinstance(macro_payload, dict):
        return macro_payload.get("status")
    if isinstance(macro_payload, list):
        statuses = [
            str(item.get("status"))
            for item in macro_payload
            if isinstance(item, dict) and item.get("status") is not None
        ]
        if not statuses:
            return None
        counts = pd.Series(statuses).value_counts()
        return ", ".join(f"{name}={count}" for name, count in counts.items())
    return None


fmp_refresh_summary_path = paths.equity_artifact_dir / "fmp_refresh_summary.json"
fmp_refresh_summary_path.parent.mkdir(parents=True, exist_ok=True)
fmp_refresh_summary_path.write_text(json.dumps(fmp_refresh_summary, indent=2, default=str), encoding="utf-8")
print(f"Wrote FMP refresh summary to {fmp_refresh_summary_path}")
display(pd.DataFrame([{
    "equity_symbols": len(fmp_refresh_summary.get("equity_symbols", [])),
    "etf_symbols": len(fmp_refresh_summary.get("etf_symbols", [])),
    "equity_price_total": fmp_refresh_summary.get("equity_prices", {}).get("total"),
    "equity_price_updated": fmp_refresh_summary.get("equity_prices", {}).get("updated"),
    "equity_fundamental_total": fmp_refresh_summary.get("equity", {}).get("total"),
    "equity_fundamental_updated": fmp_refresh_summary.get("equity", {}).get("updated"),
    "macro_status": macro_refresh_status(fmp_refresh_summary.get("macro")),
}]))


## Target Engineering

Build collapsed oracle trade labels from refreshed warehouse prices. This notebook uses `oracle_only` labels: side-specific `YE` oracle entries for `k=1..12`, collapsed into `oracle_long` and `oracle_short` classes.


In [ ]:
from quant_warehouse.research_tools import BinaryTargetConfig
from quant_warehouse.platforms.data_providers.fmp.target_engineering import LabelBuildSpec, build_trade_results
from quant_warehouse.warehouse.api import Warehouse
from quant_orchestrator.research_tools.ml_trading_experiment import _build_oracle_trade_label_rows_sparse

if "screened_equity_symbols" not in globals():
    raise RuntimeError("Run the FMP Historical Refresh cell before target engineering.")

target_warehouse = Warehouse()

target_config = BinaryTargetConfig(
    provider="fmp",
    start_date=DATA_START,
    end_date=optional_date(DATA_END),
    oracle_trade_k_by_frequency={"YE": tuple(range(1, 13))},
)

oracle_label_rows, oracle_label_diagnostics, oracle_seconds, oracle_trade_windows_unique = _build_oracle_trade_label_rows_sparse(
    screened_equity_symbols,
    target_config,
    warehouse=target_warehouse,
)

oracle_price_frames = {}
for raw_symbol in screened_equity_symbols:
    symbol = str(raw_symbol).strip().upper()
    if not symbol:
        continue
    prices = target_warehouse.read_prices(
        symbol,
        provider=target_config.provider,
        start=target_config.start_date,
        end=target_config.end_date,
    )
    if prices is not None and not prices.empty:
        oracle_price_frames[symbol] = prices

oracle_trade_spec = LabelBuildSpec(
    k_params={frequency: list(ks) for frequency, ks in target_config.oracle_trade_k_by_frequency.items()},
    min_profit_pct=target_config.oracle_trade_min_profit_pct,
    buy_execution=target_config.oracle_trade_long_entry_price_col,
    sell_execution=target_config.oracle_trade_long_exit_price_col,
    short_execution=target_config.oracle_trade_short_entry_price_col,
    cover_execution=target_config.oracle_trade_short_exit_price_col,
    start_date=target_config.start_date,
    end_date=target_config.end_date,
)
oracle_trade_result = build_trade_results(
    screened_equity_symbols,
    spec=oracle_trade_spec,
    price_frames=oracle_price_frames,
)
oracle_trade_windows = pd.DataFrame(oracle_trade_result.completed_trades)
if not oracle_trade_windows.empty:
    oracle_trade_windows["symbol"] = oracle_trade_windows["symbol"].astype(str).str.upper()
    oracle_trade_windows["entry_date"] = pd.to_datetime(oracle_trade_windows["entry_date"], errors="coerce").dt.normalize()
    oracle_trade_windows["exit_date"] = pd.to_datetime(oracle_trade_windows["exit_date"], errors="coerce").dt.normalize()
    oracle_trade_windows = oracle_trade_windows.dropna(subset=["symbol", "entry_date", "exit_date"])
    if "trade_id" not in oracle_trade_windows.columns:
        oracle_trade_windows["trade_id"] = (
            oracle_trade_windows["symbol"].astype(str)
            + "|"
            + oracle_trade_windows["side"].astype(str)
            + "|"
            + oracle_trade_windows["entry_date"].dt.strftime("%Y%m%d")
            + "|"
            + oracle_trade_windows["exit_date"].dt.strftime("%Y%m%d")
            + "|k"
            + oracle_trade_windows["k"].astype(str)
        )
    oracle_trade_windows = oracle_trade_windows.sort_values(
        ["entry_date", "symbol", "trade_id"],
        ascending=[False, True, True],
        kind="stable",
    ).reset_index(drop=True)

print(
    {
        "oracle_label_rows": len(oracle_label_rows),
        "oracle_classes": sorted(oracle_label_rows["collapsed_label"].dropna().unique())
        if not oracle_label_rows.empty
        else [],
        "oracle_seconds": round(oracle_seconds, 3),
        "oracle_trade_windows": len(oracle_trade_windows),
    }
)
display(oracle_label_diagnostics)
display(oracle_label_rows.head(20))

## ThetaData Oracle Entry/Exit Backfill

Backfill ThetaData EOD option chains only for oracle trade entry and exit dates from the current in-memory target-engineering table, newest entries first. Each endpoint date is checked against the Arctic cache before any ThetaData request is made.


In [ ]:
from quant_warehouse.migrate.backfill_thetadata_options import normalize_oracle_trade_windows

if "oracle_trade_windows" not in globals() or oracle_trade_windows.empty:
    raise RuntimeError("Run the Target Engineering cell before ThetaData oracle entry/exit backfill.")

required_oracle_trade_cols = {"symbol", "entry_date", "exit_date"}
missing_oracle_trade_cols = sorted(required_oracle_trade_cols.difference(oracle_trade_windows.columns))
if missing_oracle_trade_cols:
    raise RuntimeError(
        "The in-memory oracle_trade_windows table does not contain the columns needed for ThetaData entry/exit refresh: "
        f"{missing_oracle_trade_cols}"
    )

oracle_backfill_symbols = tuple(screened_equity_symbols) if "screened_equity_symbols" in globals() else ()
theta_oracle_trade_windows_raw = oracle_trade_windows.copy()
theta_oracle_trade_windows = normalize_oracle_trade_windows(
    theta_oracle_trade_windows_raw,
    symbols=oracle_backfill_symbols,
)
if theta_oracle_trade_windows.empty:
    raise RuntimeError("No normalized oracle trade windows are available for ThetaData entry/exit refresh.")

endpoint_pairs = pd.concat(
    [
        theta_oracle_trade_windows[["symbol", "entry_date"]].rename(columns={"entry_date": "snapshot_date"}),
        theta_oracle_trade_windows[["symbol", "exit_date"]].rename(columns={"exit_date": "snapshot_date"}),
    ],
    ignore_index=True,
)
endpoint_pairs["symbol"] = endpoint_pairs["symbol"].astype(str).str.upper()
endpoint_pairs["snapshot_date"] = pd.to_datetime(endpoint_pairs["snapshot_date"], errors="coerce").dt.normalize()
endpoint_pairs = endpoint_pairs.dropna(subset=["symbol", "snapshot_date"]).drop_duplicates()

theta_backfill_shape = {
    "raw_oracle_trade_rows": len(theta_oracle_trade_windows_raw),
    "deduped_symbol_entry_exit_windows": len(theta_oracle_trade_windows),
    "unique_symbol_endpoint_dates": len(endpoint_pairs),
    "symbols": int(theta_oracle_trade_windows["symbol"].nunique()),
}
print("ThetaData oracle entry/exit source: in-memory oracle_trade_windows")
display(pd.DataFrame([theta_backfill_shape]))
print(
    "ThetaData oracle entry/exit mode checks the Arctic cache first; "
    "ThetaData is called only for missing unique symbol/date entry-exit endpoints."
)

thetadata_oracle_trade_window_summary = {"status": "skipped_by_config"}
if REFRESH_THETADATA_ORACLE_TRADE_WINDOWS:
    thetadata_oracle_trade_window_summary = backfill_thetadata_for_oracle_trade_windows(
        theta_oracle_trade_windows,
        symbols=oracle_backfill_symbols,
        backfill_window_days=THETADATA_ORACLE_BACKFILL_WINDOW_DAYS,
        fallback_window_days=THETADATA_ORACLE_FALLBACK_WINDOW_DAYS,
        overwrite=THETADATA_ORACLE_BACKFILL_OVERWRITE,
        request_sleep=THETADATA_ORACLE_BACKFILL_REQUEST_SLEEP,
    )
else:
    print("ThetaData oracle entry/exit backfill skipped because REFRESH_THETADATA_ORACLE_TRADE_WINDOWS=False")

display(thetadata_oracle_trade_window_summary)


## Feature Engineering

Build the point-in-time feature-family panel from the refreshed Quant Warehouse store. This uses the screened FMP universe and keeps only the strategy sources configured for the equity MoE.


In [ ]:
from quant_warehouse.research_tools import (
    FamilyEvaluationConfig,
    build_fundamental_feature_panel,
    cap_features_by_quality,
)
from quant_warehouse.warehouse.api import Warehouse

if "screened_equity_symbols" not in globals():
    raise RuntimeError("Run the FMP Historical Refresh cell before feature engineering.")

engineering_warehouse = Warehouse()
feature_config = FamilyEvaluationConfig(
    provider="fmp",
    market_cap_min=MIN_MARKET_CAP,
    start_date=DATA_START,
    end_date=optional_date(DATA_END),
)

raw_feature_panel, raw_feature_metadata, feature_diagnostics, feature_timings = build_fundamental_feature_panel(
    screened_equity_symbols,
    feature_config,
    warehouse=engineering_warehouse,
)
selected_features, selected_feature_metadata, feature_quality = cap_features_by_quality(
    raw_feature_panel,
    raw_feature_metadata,
    max_features=None,
)
wanted_sources = {str(source).strip() for source in DEFAULT_STRATEGY_SOURCES}
raw_metadata = raw_feature_metadata.copy()
raw_metadata["strategy_source"] = raw_metadata["source"].astype(str) + "." + raw_metadata["family"].astype(str)
raw_available_sources = set(raw_metadata["strategy_source"].astype(str))
missing_raw_sources = sorted(wanted_sources.difference(raw_available_sources))
metadata = selected_feature_metadata.copy()
metadata["strategy_source"] = metadata["source"].astype(str) + "." + metadata["family"].astype(str)
selected_feature_metadata = (
    metadata.loc[metadata["strategy_source"].isin(wanted_sources)]
    .drop(columns=["strategy_source"])
    .reset_index(drop=True)
)
selected_available_sources = set((selected_feature_metadata["source"].astype(str) + "." + selected_feature_metadata["family"].astype(str)))
missing_selected_sources = sorted(wanted_sources.difference(selected_available_sources))
if REQUIRE_ALL_REQUESTED_FEATURE_FAMILIES and (missing_raw_sources or missing_selected_sources):
    raise RuntimeError(
        "Requested feature families are missing and equity ML training is configured to require all of them. "
        f"missing_raw={missing_raw_sources}; missing_selected={missing_selected_sources}"
    )
selected_features = [
    feature
    for feature in selected_features
    if feature in set(selected_feature_metadata["feature"].astype(str))
]
feature_panel = raw_feature_panel[["symbol", "date", *selected_features]].copy()
feature_panel["symbol"] = feature_panel["symbol"].astype(str).str.upper()
feature_panel["date"] = pd.to_datetime(feature_panel["date"], errors="coerce").dt.normalize()

print(
    {
        "symbols": len(screened_equity_symbols),
        "raw_feature_panel_rows": len(raw_feature_panel),
        "raw_feature_columns": len(raw_feature_panel.columns) if not raw_feature_panel.empty else 0,
        "selected_feature_columns": len(selected_features),
        "selected_feature_families": selected_feature_metadata["family"].nunique(),
        "requested_feature_families": len(wanted_sources),
        "missing_requested_raw_families": missing_raw_sources,
        "missing_requested_selected_families": missing_selected_sources,
        **feature_timings,
    }
)
display(feature_diagnostics.head(20))
display(
    selected_feature_metadata.groupby(["source", "family"], as_index=False)
    .agg(feature_count=("feature", "nunique"))
    .sort_values(["source", "family"])
)


## Equity MoE Training

Train RAPIDS/cuML random-forest classifiers on the refreshed warehouse data and engineered feature/target panels. This uses the screened `MIN_MARKET_CAP = 1_000_000_000_000` universe, fits on all available labeled rows from `1900-01-01` through the latest stored data, and writes the standard live-ranking artifacts. Internal orchestrator refresh, model probability diagnostics, and trading-validation diagnostics are disabled for this live-training path.


In [ ]:
from dataclasses import fields
from inspect import signature
import importlib
import sys
from pathlib import Path
from time import perf_counter
from types import SimpleNamespace

if "WORKSPACE_ROOT" not in globals():
    _repo_root_for_import = Path.cwd().resolve()
    while not (_repo_root_for_import / "app").is_dir() and _repo_root_for_import != _repo_root_for_import.parent:
        _repo_root_for_import = _repo_root_for_import.parent
    WORKSPACE_ROOT = _repo_root_for_import.parent


def force_local_repo_import(package_name: str, repo_dir: Path) -> None:
    repo_dir = repo_dir.expanduser().resolve()
    if not repo_dir.exists():
        raise RuntimeError(f"Expected local repo for {package_name!r} at {repo_dir}, but it does not exist")
    repo_text = str(repo_dir)
    sys.path[:] = [
        entry
        for entry in sys.path
        if str(Path(entry or ".").expanduser().resolve()) != repo_text
    ]
    sys.path.insert(0, repo_text)
    importlib.invalidate_caches()
    loaded = sys.modules.get(package_name)
    loaded_file = Path(str(getattr(loaded, "__file__", ""))).resolve() if loaded is not None else None
    if loaded_file is not None and repo_dir not in loaded_file.parents:
        for module_name in [name for name in sys.modules if name == package_name or name.startswith(f"{package_name}.")]:
            del sys.modules[module_name]


force_local_repo_import("quant_orchestrator", WORKSPACE_ROOT / "quant-orchestrator")
force_local_repo_import("quant_warehouse", WORKSPACE_ROOT / "quant-warehouse")

import quant_orchestrator.research_tools.ml_trading_experiment as ml_exp
import quant_warehouse.research_tools as qw_tools
from quant_orchestrator.platforms.ml_frameworks.rapids.random_forest import RapidsRandomForestClassifier
from quant_orchestrator.research_tools.ml_trading import write_ml_trading_artifact_files
from quant_orchestrator.research_tools.ml_trading_experiment import (
    MLTradingExperimentConfig,
    _score_family_models,
    _train_family_models,
)


required_ml_config_fields = {
    "symbols",
    "score_start",
    "fit_all_available_data",
    "strategy_sources",
    "feature_representations",
    "ae_config",
    "target_label_mode",
    "run_model_diagnostics",
    "run_trading_diagnostics",
}
available_ml_config_fields = {field.name for field in fields(MLTradingExperimentConfig)}
missing_ml_config_fields = sorted(required_ml_config_fields.difference(available_ml_config_fields))
train_signature = signature(_train_family_models)
if missing_ml_config_fields or "run_model_diagnostics" not in train_signature.parameters:
    raise RuntimeError(
        "The notebook imported an older quant_orchestrator instead of the local source checkout. "
        f"module={getattr(ml_exp, '__file__', None)!r}; "
        f"missing_config_fields={missing_ml_config_fields}; "
        "restart the kernel and run the bootstrap cell so the local quant-orchestrator repo is first on sys.path."
    )


def notebook_training_log(message: str) -> None:
    print(f"[moe-train] {message}", flush=True)


def _restore_training_log_hooks(hooks: dict[str, object]) -> None:
    qw_tools.build_fundamental_feature_panel = hooks["build_fundamental_feature_panel"]
    ml_exp._build_oracle_trade_label_rows_sparse = hooks["build_oracle_trade_label_rows_sparse"]
    RapidsRandomForestClassifier.fit = hooks["rapids_fit"]


def _install_training_log_hooks(planned_models: list[str]) -> dict[str, object]:
    hooks = {
        "build_fundamental_feature_panel": qw_tools.build_fundamental_feature_panel,
        "build_oracle_trade_label_rows_sparse": ml_exp._build_oracle_trade_label_rows_sparse,
        "rapids_fit": RapidsRandomForestClassifier.fit,
    }
    fit_state = {"i": 0}
    total_planned = len(planned_models)

    def _wrap_phase(phase_name: str, fn):
        def wrapped(*args, **kwargs):
            notebook_training_log(f"Starting {phase_name}")
            started = perf_counter()
            result = fn(*args, **kwargs)
            notebook_training_log(f"Finished {phase_name} in {perf_counter() - started:.1f}s")
            return result

        return wrapped

    @classmethod
    def _logged_rf_fit(cls, frame, *, features, target_col, random_state, params):
        fit_state["i"] += 1
        job_index = fit_state["i"]
        model_name = planned_models[job_index - 1] if job_index <= total_planned else f"model_{job_index}"
        notebook_training_log(
            f"[{job_index}/{total_planned}] Fitting {model_name} | "
            f"estimator=RapidsRandomForestClassifier autoencoder=disabled "
            f"rows={len(frame):,} features={len(features)}"
        )
        started = perf_counter()
        model = hooks["rapids_fit"](
            frame,
            features=features,
            target_col=target_col,
            random_state=random_state,
            params=params,
        )
        notebook_training_log(
            f"[{job_index}/{total_planned}] Finished {model_name} in {perf_counter() - started:.1f}s"
        )
        return model

    qw_tools.build_fundamental_feature_panel = _wrap_phase(
        "feature_panel rebuild inside orchestrator",
        hooks["build_fundamental_feature_panel"],
    )
    ml_exp._build_oracle_trade_label_rows_sparse = _wrap_phase(
        "oracle label rebuild inside orchestrator",
        hooks["build_oracle_trade_label_rows_sparse"],
    )
    RapidsRandomForestClassifier.fit = _logged_rf_fit
    return hooks


required_training_inputs = ["screened_equity_symbols", "feature_panel", "selected_feature_metadata", "oracle_label_rows"]
missing_training_inputs = [name for name in required_training_inputs if name not in globals()]
if missing_training_inputs:
    raise RuntimeError(
        "Run the FMP Historical Refresh, Feature Engineering, and Target Engineering cells before training. "
        f"Missing: {missing_training_inputs}"
    )

latest_feature_score_start = DATA_START
if "feature_panel" in globals() and not feature_panel.empty:
    latest_feature_date = pd.to_datetime(feature_panel["date"], errors="coerce").max()
    if pd.notna(latest_feature_date):
        latest_feature_score_start = latest_feature_date.strftime("%Y-%m-%d")

train_end_value = optional_date(DATA_END) or str(pd.Timestamp.today().date())
oos_start_value = (pd.Timestamp(train_end_value) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
EXPECTED_MIN_MARKET_CAP = int(MIN_MARKET_CAP)

equity_config = MLTradingExperimentConfig(
    experiment_name="trading_app_v2_10b_oracle_ye_moe",
    mode="classifier",
    min_market_cap=MIN_MARKET_CAP,
    symbols=tuple(screened_equity_symbols),
    start_date=DATA_START,
    end_date=optional_date(DATA_END),
    train_end=train_end_value,
    oos_start=oos_start_value,
    score_start=latest_feature_score_start,
    fit_all_available_data=True,
    refresh_missing_fmp_data=False,
    refresh_missing_fmp_include_macro=REFRESH_MISSING_FMP_INCLUDE_MACRO,
    refresh_missing_fmp_include_prices=REFRESH_MISSING_FMP_INCLUDE_PRICES,
    top_k_values=(TOP_K,),
    strategy_sources=tuple(DEFAULT_STRATEGY_SOURCES),
    feature_representations=("raw",),
    ae_config=None,
    target_label_mode="oracle_only",
    oracle_frequencies=("YE",),
    oracle_k_min=1,
    oracle_k_max=12,
    quant_warehouse_root=None,
    run_zipline_backtests=RUN_BACKTESTS_IN_ORCHESTRATOR,
    include_yearly_vectorized_diagnostics=False,
    backtesting_py_symbol_cases_per_side=0,
    run_model_diagnostics=False,
    run_trading_diagnostics=False,
    log_mlflow=False,
)

equity_config_display = asdict(equity_config)
display(equity_config_display)

assert equity_config.mode == "classifier"
assert equity_config.feature_representations == ("raw",)
assert equity_config.ae_config is None
assert equity_config.start_date == "1900-01-01"
assert equity_config.fit_all_available_data is True
assert equity_config.refresh_missing_fmp_data is False
assert equity_config.symbols == tuple(screened_equity_symbols)
assert equity_config.min_market_cap == EXPECTED_MIN_MARKET_CAP
assert equity_config.score_start == latest_feature_score_start
assert equity_config.top_k_values == (TOP_K,)
assert equity_config.run_model_diagnostics is False
assert equity_config.run_trading_diagnostics is False

training_guard = pd.DataFrame(
    [
        {
            "training_path": "classifier_only_live",
            "estimator": "RapidsRandomForestClassifier",
            "mode": equity_config.mode,
            "feature_representation": ", ".join(equity_config.feature_representations),
            "autoencoder": "disabled",
            "ae_config": equity_config.ae_config,
            "model_probability_diagnostics": equity_config.run_model_diagnostics,
            "trading_validation_diagnostics": equity_config.run_trading_diagnostics,
            "zipline_backtests": equity_config.run_zipline_backtests,
            "yearly_vectorized_diagnostics": equity_config.include_yearly_vectorized_diagnostics,
            "backtesting_py_cases_per_side": equity_config.backtesting_py_symbol_cases_per_side,
        }
    ]
)
print("Training execution guard")
display(training_guard)

actual_training_families = (
    selected_feature_metadata[["source", "family"]]
    .drop_duplicates()
    .assign(strategy_source=lambda frame: frame["source"].astype(str) + "." + frame["family"].astype(str))
    .sort_values(["source", "family"])
    .reset_index(drop=True)
)
planned_models = actual_training_families["strategy_source"].tolist()
requested_sources = set(str(source).strip() for source in DEFAULT_STRATEGY_SOURCES)
missing_requested_sources = sorted(requested_sources.difference(planned_models))
unexpected_training_sources = sorted(set(planned_models).difference(requested_sources))
if REQUIRE_ALL_REQUESTED_FEATURE_FAMILIES:
    assert not missing_requested_sources, missing_requested_sources
    assert not unexpected_training_sources, unexpected_training_sources
training_plan = pd.DataFrame(
    {
        "job": range(1, len(planned_models) + 1),
        "strategy_source": planned_models,
        "estimator": "RapidsRandomForestClassifier",
        "model_type": "classifier",
        "autoencoder": "disabled",
        "ae_config": equity_config.ae_config,
        "representation": equity_config.feature_representations[0],
        "model_probability_diagnostics": equity_config.run_model_diagnostics,
        "trading_validation_diagnostics": equity_config.run_trading_diagnostics,
    }
)
print("Classifier-only models queued for training")
display(training_plan)
if missing_requested_sources:
    print("Requested strategy sources not trained because no selected feature columns were available")
    display(pd.DataFrame({"strategy_source": missing_requested_sources}))

training_log_hooks = None
if RUN_EQUITY_MOE_TRAINING:
    notebook_training_log(f"Starting classifier-only equity training with {len(planned_models)} planned classifier jobs")
    training_started = perf_counter()
    phase_rows = []

    def mark_training_phase(phase: str, **metadata):
        nonlocal_phase_started = phase_rows[-1]["elapsed_seconds"] if phase_rows else 0.0
        now_elapsed = perf_counter() - training_started
        row = {"phase": phase, "seconds": now_elapsed - nonlocal_phase_started, "elapsed_seconds": now_elapsed}
        row.update(metadata)
        phase_rows.append(row)

    training_log_hooks = _install_training_log_hooks(planned_models)
    try:
        model_results, trained_models = _train_family_models(
            equity_config,
            feature_panel,
            selected_feature_metadata,
            oracle_label_rows,
            train_end=pd.Timestamp(train_end_value),
            oos_start=pd.Timestamp(oos_start_value),
            fit_all_available_data=True,
            run_model_diagnostics=False,
        )
    finally:
        _restore_training_log_hooks(training_log_hooks)
    mark_training_phase(
        "train_family_models",
        trained_models=int((model_results["status"] == "ok").sum()) if "status" in model_results else len(model_results),
    )
    strategy_scores, mean_scores = _score_family_models(
        equity_config,
        feature_panel,
        trained_models,
        score_start=pd.Timestamp(latest_feature_score_start),
    )
    mark_training_phase(
        "score_latest_family_models",
        score_rows=len(strategy_scores),
        strategy_sources=strategy_scores["strategy_source"].nunique() if not strategy_scores.empty else 0,
    )
    phase_timings = pd.DataFrame(phase_rows)
    empty_diagnostics = pd.DataFrame()
    elapsed_seconds = perf_counter() - training_started
    equity_metrics = {
        "symbols": int(strategy_scores["symbol"].nunique()) if "symbol" in strategy_scores else 0,
        "trained_models": int((model_results["status"] == "ok").sum()) if "status" in model_results else int(len(model_results)),
        "strategy_sources": int(strategy_scores["strategy_source"].nunique()) if "strategy_source" in strategy_scores else 0,
        "best_sharpe": None,
        "best_total_return": None,
        "best_max_drawdown": None,
        "elapsed_seconds": float(elapsed_seconds),
    }
    equity_result = SimpleNamespace(
        model_results=model_results,
        strategy_scores=strategy_scores,
        backtest_summary=empty_diagnostics,
        trade_log=empty_diagnostics,
        model_vs_trading=empty_diagnostics,
        metric_correlations=empty_diagnostics,
        yearly_backtest_summary=empty_diagnostics,
        symbol_strategy_summary=empty_diagnostics,
        symbol_robustness_summary=empty_diagnostics,
        backtesting_py_symbol_validation=empty_diagnostics,
        phase_timings=phase_timings,
        analysis_markdown="Classifier-only live training: model diagnostics and trading validation were skipped.",
        metrics=equity_metrics,
    )
    notebook_training_log("Classifier-only equity training finished")
    if not equity_result.phase_timings.empty:
        print("Experiment phase timings")
        display(equity_result.phase_timings)
    write_ml_trading_artifact_files(
        model_results=equity_result.model_results,
        strategy_scores=equity_result.strategy_scores,
        backtest_summary=equity_result.backtest_summary,
        trade_log=equity_result.trade_log,
        model_vs_trading=equity_result.model_vs_trading,
        metric_correlations=equity_result.metric_correlations,
        yearly_backtest_summary=equity_result.yearly_backtest_summary,
        symbol_strategy_summary=equity_result.symbol_strategy_summary,
        symbol_robustness_summary=equity_result.symbol_robustness_summary,
        backtesting_py_symbol_validation=equity_result.backtesting_py_symbol_validation,
        phase_timings=equity_result.phase_timings,
        analysis_markdown=equity_result.analysis_markdown,
        directory=paths.equity_artifact_dir,
    )
    live_config = asdict(equity_config)
    live_config["quant_warehouse_storage"] = storage.as_dict()
    live_config["fmp_refresh_summary_path"] = str(fmp_refresh_summary_path)
    (paths.equity_artifact_dir / "live_config.json").write_text(
        json.dumps(live_config, indent=2, default=str),
        encoding="utf-8",
    )
    print(equity_result.metrics)

if "equity_result" not in globals():
    raise RuntimeError(
        "No in-memory equity_result is available for this notebook run. "
        "Set RUN_EQUITY_MOE_TRAINING=True and rerun the Equity MoE Training cell; "
        "the notebook no longer reloads stale strategy_scores/model_results CSV files."
    )

strategy_scores = equity_result.strategy_scores.copy()
model_results = equity_result.model_results.copy()
backtest_summary = equity_result.backtest_summary.copy()

model_visibility_cols = [
    col
    for col in [
        "strategy_source",
        "source",
        "family",
        "base_family",
        "representation",
        "status",
        "features",
        "raw_features",
        "rows",
        "train_rows",
        "oos_rows",
        "training_window",
        "classifier_backend",
        "gpu_device_id",
        "gpu_device_name",
        "cudf_version",
        "cuml_version",
        "cupy_version",
        "classes",
        "train_accuracy",
        "oos_accuracy",
        "oos_macro_f1",
        "oos_balanced_accuracy",
    ]
    if col in model_results.columns
]

print("Actual classifier model rows")
display(model_results[model_visibility_cols].head(50) if model_visibility_cols else model_results.head(50))
print("Strategy score columns")
display(pd.DataFrame({"column": strategy_scores.columns.tolist()}))
print("Latest strategy scores sample")
display(strategy_scores.head())
print("Backtest summary sample")
display(backtest_summary.head())


## Strategy Configuration

Review the 10B universe training configuration after model fitting: requested feature families, oracle `YE` labels for `k=1..12`, classifier jobs, and GPU preflight metadata.


In [ ]:
feature_family_plan = pd.DataFrame(
    [
        {
            "strategy_source": name,
            "provider": name.split(".", 1)[0],
            "feature_family": name.split(".", 1)[1],
        }
        for name in DEFAULT_STRATEGY_SOURCES
    ]
)

oracle_label_plan = pd.DataFrame(
    [
        {
            "oracle_frequency": "YE",
            "k": k,
            "source_target_column": f"target_oracle_trade_entry__YE_k{k}_{side}",
            "classifier_label": "oracle_long" if side == "long" else "oracle_short" if side == "short" else "not a classifier class",
            "used_by_classifier": side in {"long", "short"},
        }
        for k in range(1, 13)
        for side in ("long", "short", "any")
    ]
)

classifier_plan = feature_family_plan.assign(
    classifier="RapidsRandomForestClassifier",
    backend="rapids_cuml_gpu",
    target_column="collapsed_label",
    classifier_labels="oracle_long, oracle_short",
    feature_representation="raw",
)

from quant_orchestrator.platforms.ml_frameworks.rapids.random_forest import rapids_gpu_info

gpu_preflight = rapids_gpu_info()

print("Feature families requested")
display(feature_family_plan)
print("Oracle label columns requested from YE k=1..12")
display(oracle_label_plan)
print("Classifier jobs planned")
display(classifier_plan)
print("RAPIDS GPU preflight")
display(gpu_preflight)


## Live Leaderboard

The leaderboard converts the current in-memory `strategy_scores` table into a live ranking. Latest prices are read from `quant-warehouse`, not from local `optimal_trader` data builders.

In [ ]:
leaderboard = build_latest_equity_leaderboard(
    strategy_scores,
    top_k=TOP_K,
    min_long_score=MIN_LONG_SCORE,
    price_provider="fmp",
)
display(leaderboard.head(TOP_K + 5))

## Option Ranker Training

This cell shows the direct `quant-orchestrator` option-family ranker call. It only reads the existing ThetaData-derived option candidate panel; it does not refresh or rebuild ThetaData option chains. The ranker joins FMP/FinanceToolkit feature-family panels from the shared `quant-warehouse` store and writes selector/family ranking artifacts for the option strategy.


In [ ]:
from quant_orchestrator.research_tools import (
    OptionFamilyRankerConfig,
    run_option_family_ranker_experiment,
)

option_train_end_value = optional_date(DATA_END) or "2024-12-31"
option_eval_start_value = (pd.Timestamp(option_train_end_value) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
# Split dates are retained for compatibility but ignored while train_on_all_data=True.

option_ranker_config = OptionFamilyRankerConfig(
    option_panel=OPTION_PANEL_FOR_RANKER.expanduser().resolve(),
    output_dir=paths.option_artifact_dir,
    min_market_cap=MIN_MARKET_CAP,
    start_date=DATA_START,
    end_date=optional_date(DATA_END),
    train_end=option_train_end_value,
    eval_start=option_eval_start_value,
    train_on_all_data=True,
    target_col="rank_y",
    all_feature_families=False,
    feature_families=tuple(DEFAULT_STRATEGY_SOURCES),
    n_estimators=400,
    max_trades=0,
    random_seed=20260704,
    disable_pairwise_ranker=True,
)

display(asdict(option_ranker_config))

if RUN_OPTION_RANKER_TRAINING:
    if not OPTION_PANEL_FOR_RANKER.exists():
        raise FileNotFoundError(
            f"Missing option panel {OPTION_PANEL_FOR_RANKER}. ThetaData refresh/build is intentionally disabled here; point OPTION_PANEL_FOR_RANKER at an existing candidate panel."
        )
    option_ranker_result = run_option_family_ranker_experiment(option_ranker_config)
    print(option_ranker_result.output_dir)
    display(option_ranker_result.family_summary.head(20))
else:
    print("Option ranker training skipped; not reading old option ranker summary CSV files.")


## ThetaData Score-Date EOD Refresh

Refresh the ThetaData EOD option chain only for the current selected equity symbols and the equity score date. This keeps live option ML ranking scoped to contracts that were tradable on the score date instead of historical expired option panels.


In [ ]:
if "leaderboard" not in globals() or leaderboard.empty:
    raise RuntimeError("Run the leaderboard cell before ThetaData score-date refresh.")

theta_score_date = optional_date(THETADATA_OPTION_SCORE_DATE)
if theta_score_date is None:
    theta_score_date = pd.to_datetime(leaderboard["score_date"], errors="coerce").max().strftime("%Y-%m-%d")

theta_selected_symbols = (
    leaderboard.loc[leaderboard["selected"].astype(bool), "symbol"].astype(str).str.upper().tolist()
    if "selected" in leaderboard.columns
    else leaderboard["symbol"].astype(str).str.upper().tolist()
)
print(f"ThetaData score-date EOD date: {theta_score_date}")
print(f"ThetaData selected symbols: {theta_selected_symbols}")

thetadata_score_date_refresh_summary = None
if REFRESH_THETADATA_SCORE_DATE_EOD:
    thetadata_score_date_refresh_summary = backfill_thetadata_eod_for_score_date(
        symbols=theta_selected_symbols,
        score_date=theta_score_date,
        max_workers=THETADATA_REFRESH_MAX_WORKERS,
        overwrite=THETADATA_REFRESH_OVERWRITE,
    )
    display(thetadata_score_date_refresh_summary)
else:
    print("ThetaData score-date refresh skipped because REFRESH_THETADATA_SCORE_DATE_EOD=False")

score_date_option_ml_rankings = build_score_date_option_ml_ranking_table(
    paths.option_artifact_dir,
    leaderboard=leaderboard,
    score_date=theta_score_date,
    symbols=theta_selected_symbols,
    target_dte=OPTION_TENOR_DAYS,
    min_market_cap=MIN_MARKET_CAP,
    start_date=DATA_START,
)
print(f"Score-date option ML ranking rows: {len(score_date_option_ml_rankings)}")
display(score_date_option_ml_rankings)


## Build Broker Order Plans

Three Alpaca paper accounts are managed separately:

- equity paper: equity variant of the feature-family MoE strategy
- option paper: option variant of the same equity signals
- LLM paper: top trades sent to `trading_agents` for review before paper execution

Robinhood real orders are based on the option strategy and use GTC limit orders. Option buys are priced at the bid and option sells are priced at the ask; the Robinhood gate controls whether new orders are allowed.

In [ ]:
paper_order_plans: dict[str, pd.DataFrame] = {}
paper_order_clients: dict[str, object] = {}
option_plan: dict[str, pd.DataFrame] = {}

try:
    equity_orders = build_alpaca_equity_orders(
        leaderboard=leaderboard,
        account_prefix=ALPACA_EQUITY_ACCOUNT_PREFIX,
        gross_exposure=0.95,
    )
    paper_order_plans["alpaca_equity_paper"] = equity_orders
    paper_order_clients["alpaca_equity_paper"] = alpaca_client_from_env(ALPACA_EQUITY_ACCOUNT_PREFIX)
except Exception as exc:
    print(f"Alpaca equity paper plan skipped: {type(exc).__name__}: {exc}")

try:
    option_plan = build_alpaca_option_orders(
        leaderboard=leaderboard,
        account_prefix=ALPACA_OPTION_ACCOUNT_PREFIX,
        strategy_allocation=OPTION_STRATEGY_ALLOCATION,
        option_bucket=OPTION_BUCKET,
        tenor_days=OPTION_TENOR_DAYS,
    )
    paper_order_plans["alpaca_option_paper"] = option_plan.get("actionable_orders", pd.DataFrame())
    paper_order_clients["alpaca_option_paper"] = option_plan["client"]
except Exception as exc:
    print(f"Alpaca option paper plan skipped: {type(exc).__name__}: {exc}")

try:
    llm_orders = build_llm_review_orders(
        leaderboard=leaderboard,
        top_k=TOP_K,
        account_prefix=ALPACA_LLM_ACCOUNT_PREFIX,
    )
    paper_order_plans["alpaca_llm_paper"] = llm_orders
    paper_order_clients["alpaca_llm_paper"] = alpaca_client_from_env(ALPACA_LLM_ACCOUNT_PREFIX)
except Exception as exc:
    print(f"Alpaca LLM paper plan skipped: {type(exc).__name__}: {exc}")

for name, frame in paper_order_plans.items():
    print(name, len(frame))
    display(frame.head(20))

In [ ]:
robinhood_option_plan: dict[str, pd.DataFrame] = {}
robinhood_option_orders = pd.DataFrame()
if option_plan and not option_plan.get("target_contracts", pd.DataFrame()).empty:
    robinhood_option_plan = build_robinhood_option_orders(
        target_contracts=option_plan.get("target_contracts", pd.DataFrame()),
        gate_discount_pct=ROBINHOOD_OPTION_GATE_DISCOUNT_PCT,
    )
    robinhood_option_orders = robinhood_option_plan.get("actionable_orders", pd.DataFrame())

if robinhood_option_plan:
    print("Robinhood option reconciliation summary")
    display(robinhood_option_plan.get("summary", pd.DataFrame()))
    print("Robinhood current option positions")
    display(robinhood_option_plan.get("current_option_positions", pd.DataFrame()).head(20))
    print("Robinhood pending option orders")
    display(robinhood_option_plan.get("pending_option_orders", pd.DataFrame()).head(20))

print("Robinhood actionable option orders")
display(robinhood_option_orders.head(20))
print({
    "robinhood_option_gate_discount_pct": ROBINHOOD_OPTION_GATE_DISCOUNT_PCT,
    "robinhood_equity_gate_discount_pct": ROBINHOOD_EQUITY_GATE_DISCOUNT_PCT,
    "robinhood_llm_gate_discount_pct": ROBINHOOD_LLM_GATE_DISCOUNT_PCT,
    "option_pricing_policy": "buy at bid, sell at ask",
})


## Save Leaderboard And Streamlit App

In [ ]:
option_ml_rankings_for_live = (
    score_date_option_ml_rankings
    if "score_date_option_ml_rankings" in globals()
    else build_option_ml_ranking_table(
        paths.option_artifact_dir,
        symbols=leaderboard.loc[leaderboard["selected"].astype(bool), "symbol"].tolist() if "selected" in leaderboard.columns else leaderboard["symbol"].tolist(),
    )
)
symbol_scores_for_live = build_symbol_score_table(strategy_scores, leaderboard)
orders_for_live = {**paper_order_plans, "robinhood_option_real": robinhood_option_orders}

import socket


def port_is_open(port: int, host: str = "127.0.0.1") -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.2)
        return sock.connect_ex((host, int(port))) == 0


def first_free_port(start: int = 8501, stop: int = 8599) -> int:
    for port in range(int(start), int(stop) + 1):
        if not port_is_open(port):
            return port
    raise RuntimeError(f"No free Streamlit port found between {start} and {stop}")


streamlit_app = write_streamlit_leaderboard_app(
    live_dir=paths.live_artifact_dir,
    leaderboard=leaderboard,
    symbol_scores=symbol_scores_for_live,
    option_ml_rankings=option_ml_rankings_for_live,
    orders=orders_for_live,
)
streamlit_port = first_free_port()
print({"streamlit_app": str(streamlit_app), "data_source": "in_memory_embedded"})
if streamlit_port != 8501:
    print(f"Port 8501 is already in use; using free port {streamlit_port} for this app.")
print(f"Run: streamlit run {streamlit_app} --server.address 127.0.0.1 --server.port {streamlit_port}")
print(f"Streamlit URL: http://localhost:{streamlit_port}")
print(f"Streamlit URL: http://127.0.0.1:{streamlit_port}")

## Submit Orders

The notebook never submits orders. Review the saved order plans in the generated Streamlit app, then use its Submit Orders button when you intentionally want to send broker orders.
